In [ ]:
import pandas as pd
import numpy as np
from scipy import stats


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from typing import Final
%cd /content/drive/MyDrive/colab mate/portfolio_2

/content/drive/MyDrive/colab mate/portfolio_2


In [ ]:
test = pd.read_csv('test_portfolio.csv')

In [ ]:
test.head()

,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-25,6166808460,Ireland,desktop,Europe,Organic Search,1,1,session_with_order,1
1,2020-11-21,5755801299,United States,desktop,Americas,Direct,3,2,session_with_order,1
2,2021-01-13,4049792310,United States,mobile,Americas,Direct,4,1,session_with_order,1
3,2020-11-18,445911836,United States,desktop,Americas,Direct,1,2,session_with_order,1
4,2020-11-03,1127037546,United States,desktop,Americas,Organic Search,2,2,session_with_order,1


In [ ]:
print(test.head())
print(test.columns)

         date  ga_session_id        country   device continent  \
0  2020-11-25     6166808460        Ireland  desktop    Europe   
1  2020-11-21     5755801299  United States  desktop  Americas   
2  2021-01-13     4049792310  United States   mobile  Americas   
3  2020-11-18      445911836  United States  desktop  Americas   
4  2020-11-03     1127037546  United States  desktop  Americas   

          channel  test  test_group          event_name  value  
0  Organic Search     1           1  session_with_order      1  
1          Direct     3           2  session_with_order      1  
2          Direct     4           1  session_with_order      1  
3          Direct     1           2  session_with_order      1  
4  Organic Search     2           2  session_with_order      1  
Index(['date', 'ga_session_id', 'country', 'device', 'continent', 'channel',
       'test', 'test_group', 'event_name', 'value'],
      dtype='object')


In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Definicja metryk konwersji
conversion_metrics = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_accounts']


# 2. Łączna liczba UNIKALNYCH sesji per KONTYNENT i GRUPA
sessions_per_group = test.groupby(['continent', 'test_group'])['ga_session_id'].nunique().reset_index(name='sessions')

# 3. OBLICZENIE KONWERSJI
test_filtered_events = test[test['event_name'].isin(conversion_metrics)]
test_wide_sessions = test_filtered_events.groupby(['continent', 'test_group', 'ga_session_id', 'event_name']).size().unstack(fill_value=0)

for metric in conversion_metrics:
    if metric not in test_wide_sessions.columns:
        test_wide_sessions[metric] = 0

test_wide_sessions = (test_wide_sessions > 0).astype(int).reset_index()

# 4. Agregacja sumy konwersji per kontynent i grupa
grouped_conversions = test_wide_sessions.groupby(['continent', 'test_group'])[conversion_metrics].sum().reset_index()

# 5. ŁĄCZENIE BAZY Z KONWERSJAMI
grouped_data = pd.merge(sessions_per_group, grouped_conversions, on=['continent', 'test_group'], how='left').fillna(0)

# 6. PRZYGOTOWANIE STRUKTURY POD TABLEAU
final_results = []
continents = grouped_data['continent'].unique()

for continent in continents:
    c_data = grouped_data[grouped_data['continent'] == continent]

    control_row = c_data[c_data['test_group'] == 1]
    test_row = c_data[c_data['test_group'] == 2]

    if not control_row.empty and not test_row.empty:
        sessions_c = control_row['sessions'].iloc[0]
        sessions_t = test_row['sessions'].iloc[0]

        for metric in conversion_metrics:
            conv_c = control_row[metric].iloc[0]
            conv_t = test_row[metric].iloc[0]

            cr_c = conv_c / sessions_c if sessions_c > 0 else 0
            cr_t = conv_t / sessions_t if sessions_t > 0 else 0

            metric_change = (cr_t - cr_c) / cr_c if cr_c > 0 else 0

            if sessions_c > 0 and sessions_t > 0:
                z_stat, p_value = proportions_ztest(
                    [conv_c, conv_t],
                    [sessions_c, sessions_t]
                )
                is_sig = p_value < 0.05
            else:
                z_stat, p_value, is_sig = None, None, False



            final_results.append({
                'Test Number': continent,
                'Metric': f"{metric} / session",
                'Conversion Rate Control': cr_c,
                'Conversion Rate Test': cr_t,
                'Metric Change %': metric_change,
                'P Value': p_value,
                'Z Stat': z_stat,

            })

final_df = pd.DataFrame(final_results)

print("\n--- Finalny DataFrame (Gotowy do Tableau) ---")
print(final_df.head(10))

# 7. ZAPIS DO PLIKU
final_df.to_csv('ab_test_tableau_ready.csv', index=False)


--- Finalny DataFrame (Gotowy do Tableau) ---
  Test Number                       Metric  Conversion Rate Control  \
0   (not set)   add_payment_info / session                 0.031381   
1   (not set)  add_shipping_info / session                 0.043933   
2   (not set)     begin_checkout / session                 0.043933   
3   (not set)       new_accounts / session                 0.073222   
4      Africa   add_payment_info / session                 0.019341   
5      Africa  add_shipping_info / session                 0.033407   
6      Africa     begin_checkout / session                 0.033407   
7      Africa       new_accounts / session                 0.080879   
8    Americas   add_payment_info / session                 0.020063   
9    Americas  add_shipping_info / session                 0.034021   

   Conversion Rate Test  Metric Change %   P Value    Z Stat  
0              0.021359        -0.319353  0.323075  0.988158  
1              0.034951        -0.204438  0.4

w dashbordzie mam tylko jeden kolor poniewaz zrobilem test dla kontynentow i wyszlo pvalue wieksze niz 0,05

https://public.tableau.com/app/profile/maciej.klysiak/viz/SIGNIFICANCE/Dashboard1?publish=yes

https://drive.google.com/file/d/1AXYt1vQEtAwczM7gpP3wYWv_ojElah34/view?usp=sharing